In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
%cd /content

# 切到你的專案資料夾
%cd /content/drive/MyDrive/LSTM_PROGRAM

Mounted at /content/drive
/content
/content/drive/MyDrive/LSTM_PROGRAM


In [2]:
!mkdir -p ~/.ssh
!cp /content/drive/MyDrive/.ssh/id_ed25519* ~/.ssh/
!chmod 700 ~/.ssh
!chmod 600 ~/.ssh/id_ed25519

!eval "$(ssh-agent -s)" && ssh-add ~/.ssh/id_ed25519
!ssh-keyscan github.com >> ~/.ssh/known_hosts
!chmod 644 ~/.ssh/known_hosts

!ssh -T git@github.com

!git config --global user.email "joemi7878@gmail.com"
!git config --global user.name "joemi78"

!pip install arch

Agent pid 792
Identity added: /root/.ssh/id_ed25519 (joemi7878@gmail.com)
# github.com:22 SSH-2.0-1e748d5
# github.com:22 SSH-2.0-1e748d5
# github.com:22 SSH-2.0-1e748d5
# github.com:22 SSH-2.0-1e748d5
# github.com:22 SSH-2.0-1e748d5
Hi joemi78! You've successfully authenticated, but GitHub does not provide shell access.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 15.1 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import os

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras import mixed_precision

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import precision_score, recall_score, f1_score

from functools import cached_property
from typing import Union, Optional, Dict

from scipy.stats import norm, t as tdist



try:
    from arch import arch_model
    HAS_ARCH = True
except Exception:
    HAS_ARCH = False


#### SET GPU
gpus = tf.config.list_physical_devices('GPU')
print("Num GPUs:", len(gpus), gpus)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✓ memory growth set")
    except RuntimeError as e:
        print("⚠️ GPU 已初始化，無法再設定 memory growth：", e)

#### SET GPU
gpus = tf.config.list_physical_devices('GPU')
print("Num GPUs:", len(gpus), gpus)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✓ memory growth set")
    except RuntimeError as e:
        print("⚠️ GPU 已初始化，無法再設定 memory growth：", e)

# （建議）再開啟 XLA 與混合精度
tf.config.optimizer.set_jit(True)
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

print("是否可用GPU:", tf.test.is_gpu_available())
print("使用中的裝置:", tf.config.list_physical_devices('GPU'))

print("✓ GPU 初始化流程完成")


# 1. 定義檔案路徑
file_paths = {
    # "bonds_day": "./filtered_output/bonds_day_clean_period.csv",
    # "bonds_hour": "./filtered_output/bonds_hour_clean_period.csv",
    # "crypto_day": "./filtered_output/crypto_day_clean_period.csv",
    # "crypto_hour": "./filtered_output/crypto_hour_clean_period.csv",
    # "others_day": "./filtered_output/others_day_clean_period.csv",
    # "others_hour": "./filtered_output/others_hour_clean_period.csv",
    "stock_day":  "./filtered_output/stock_day_fluctuation_aligned.csv",
    # "stock_hour": "./filtered_output/stock_hour_clean_period.csv"
}


def find_date_col(df):
    for col in df.columns:
        if 'date' in col.lower():
            return col
    return df.columns[0]

def read_and_clean(file):
    # ① 正确地读 CSV，不要写 (index=True)
    df = pd.read_csv(file)

    # ② 找到原始时间列名
    date_col = find_date_col(df)

    # ③ 依次尝试各种格式去解析
    parsed = False
    for fmt in [
        '%Y-%m-%d %H:%M:%S',
        '%Y/%m/%d %H:%M:%S',
        '%Y-%m-%d %H:%M',
        '%Y/%m/%d %H:%M',
    ]:
        try:
            df[date_col] = pd.to_datetime(
                df[date_col],
                format=fmt,     # 严格匹配
                errors='raise'  # 抛错就切换下一个 fmt
            )
            parsed = True
            break
        except Exception:
            continue

    # ④ 如果上面都没能解析，再宽松一把
    if not parsed:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    # ⑤ 把分钟/秒都砍掉，保留到「小时」粒度
    df[date_col] = df[date_col].dt.floor('h').dt.tz_localize(None)

    # ⑥ 把这一列重命名为 DATE
    df = df.rename(columns={date_col: 'DATE'})

    # （可选）如果你想让 DATE 作为索引：
    # df = df.set_index('DATE')

    return df



raw_dfs = {}
for name, path in file_paths.items():
    raw_dfs[name] = read_and_clean(path)
    # print(raw_dfs[name].columns)

# ===== 資料結構：取代 all_data =====
class AssetGroupLite:
    def __init__(self, name, df):
        self.name = name
        df = df.copy()
        df['DATE'] = pd.to_datetime(df['DATE'])
        self.raw = df.set_index('DATE').sort_index()

    @cached_property
    def _close_cols(self):
        return [c for c in self.raw.columns if c.endswith('_CLOSE')]

    @cached_property
    def _vol_cols(self):
        return [c for c in self.raw.columns if c.endswith('_VOLUME')]

    @cached_property
    def close_ln(self):
        if not self._close_cols:
            return self.raw.iloc[[]]
        return np.log(self.raw[self._close_cols]).rename(
            columns=lambda x: x.replace('_CLOSE', '_CLOSE_ln')
        )

    @cached_property
    def close_ln_ret(self):
        if not self._close_cols:
            return self.raw.iloc[[]]
        logp = np.log(self.raw[self._close_cols])
        return (
            logp.diff()
               .rename(columns=lambda x: x.replace('_CLOSE', '_CLOSE_ln_ret'))
               .dropna(how='all')
        )

    @cached_property
    def close_arith_ret(self):
        if not self._close_cols:
            return self.raw.iloc[[]]
        return (
            self.raw[self._close_cols].pct_change()
                .rename(columns=lambda x: x.replace('_CLOSE', '_CLOSE_arith_ret'))
                .dropna(how='all')
        )

class DataRepository:
    REQUIRED_INDEX = "DATE"

    def __init__(self, raw_dfs: dict, check_schema: bool = True):
        self.groups = {}
        for name, df in raw_dfs.items():
            if check_schema:
                assert 'DATE' in df.columns, f"{name}: 缺少 DATE 欄"
            self.groups[name] = AssetGroupLite(name, df)

    # 補上 group()，方便外部與內部呼叫
    def group(self, name: str) -> AssetGroupLite:
        if name not in self.groups:
            raise KeyError(f"Group '{name}' 不存在。可用群組：{list(self.groups.keys())}")
        return self.groups[name]

    def series(self, group: str, series_name: str) -> pd.Series:
        g = self.group(group)
        # 加上 raw → 能抓 OHLCV、IS_TRADING 等原始欄
        search_order = ['close_ln_ret', 'close_arith_ret', 'close_ln', 'raw']

        # 1) 直接命中
        for key in search_order:
            tbl = getattr(g, key)
            if series_name in tbl.columns:
                return tbl[series_name]

        # 2) 容錯：只給 base symbol，自動補候選
        base = (series_name
                .replace('_CLOSE', '')
                .replace('_OPEN', '')
                .replace('_HIGH', '')
                .replace('_LOW', '')
                .replace('_VOLUME', '')
                .replace('_IS_TRADING', '')
                .replace('_CLOSE_ln_ret', '')
                .replace('_CLOSE_arith_ret', '')
                .replace('_CLOSE_ln', ''))
        candidates = [
            f'{base}_CLOSE_ln_ret',
            f'{base}_CLOSE_arith_ret',
            f'{base}_CLOSE_ln',
            f'{base}_OPEN',
            f'{base}_HIGH',
            f'{base}_LOW',
            f'{base}_CLOSE',
            f'{base}_VOLUME',
            f'{base}_IS_TRADING'
        ]
        for cand in candidates:
            for key in search_order:
                tbl = getattr(g, key)
                if cand in tbl.columns:
                    return tbl[cand]

        raise KeyError(f"{group}: 找不到 {series_name} 或候選 {candidates}")

    # 取整張表
    def table(self, group: str, table_name: str) -> pd.DataFrame:
        g = self.group(group)
        if not hasattr(g, table_name):
            raise KeyError(f"{group}: 無表 '{table_name}'。可用表：['close_ln_ret','close_arith_ret','close_ln']")
        return getattr(g, table_name)

    # 若要直接拿 raw 的原始價/量欄位（例如 *_CLOSE 或 *_VOLUME）
    def raw_series(self, group: str, raw_col: str) -> pd.Series:
        g = self.group(group)
        if raw_col not in g.raw.columns:
            raise KeyError(f"{group}: raw 中沒有欄位 {raw_col}")
        return g.raw[raw_col]

repo = DataRepository(raw_dfs)

####################################
#########     LSTM     #############
####################################
# ------------------ Features ------------------
def build_feature_df(repo: DataRepository, group: str, symbol: str) -> pd.DataFrame:
    s_open  = repo.raw_series(group, f'{symbol}_OPEN').asfreq('D')
    s_high  = repo.raw_series(group, f'{symbol}_HIGH').asfreq('D')
    s_low   = repo.raw_series(group, f'{symbol}_LOW').asfreq('D')
    s_close = repo.raw_series(group, f'{symbol}_CLOSE').asfreq('D')
    s_vol   = repo.raw_series(group, f'{symbol}_VOLUME').asfreq('D')
    s_flag = repo.raw_series(group, f'{symbol}_IS_TRADING').asfreq('D')
    s_lnrt  = repo.series(group, f'{symbol}_CLOSE_ln_ret').asfreq('D')
    s_ln  = repo.series(group, f'{symbol}_CLOSE_ln').asfreq('D')
    df = pd.concat([
        s_lnrt.rename(f'{symbol}_LN_RET'),
        s_open.rename(f'{symbol}_OPEN'),
        s_high.rename(f'{symbol}_HIGH'),
        s_low.rename(f'{symbol}_LOW'),
        s_close.rename(f'{symbol}_CLOSE'),
        s_vol.rename(f'{symbol}_VOLUME'),
        s_flag.rename(f'{symbol}_IS_TRADING'),
        s_ln.rename(f'{symbol}_CLOSE_LN')
        ], axis=1)
    df = df.apply(pd.to_numeric, errors='coerce')
    return df.dropna(how='any')


# ------------------ Model ------------------
###############  LSTM
def build_small_lstm(input_len: int, n_features: int) -> tf.keras.Model:
    inp = layers.Input(shape=(input_len, n_features))
    x = layers.LSTM(LSTM_UNITS, return_sequences=False)(inp)
    x = layers.Dropout(DROPOUT)(x)
    out = layers.Dense(1, activation='linear')(x)
    m = models.Model(inp, out)
    # loss_fn = 'mse'
    m.compile(optimizer=optimizers.Adam(learning_rate=LR), loss='mse')  # ← 改用 MSE（或 Huber(delta≈0.05)）
    return m

############### mcHARCH
def fit_vol_per_window(ret_window, mode='garch'):
    y = ret_window.dropna().astype(float)
    if len(y) < 30:
        lam = 0.94
        ewma_var = y.pow(2).ewm(alpha=1-lam, adjust=False).mean()
        log_sigma_series = 0.5 * np.log(np.maximum(ewma_var.values, 1e-12))
        log_sigma_series = pd.Series(log_sigma_series, index=ewma_var.index)\
                              .reindex(ret_window.index).ffill().bfill()
        sigma_next = np.sqrt(lam * ewma_var.iloc[-1] + (1-lam) * y.iloc[-1]**2)
        return log_sigma_series, float(sigma_next)

    vol = 'HARCH' if mode.lower() == 'harch' else 'GARCH'
    p, q = (3, 0) if vol == 'HARCH' else (1, 1)

    scale = 100.0
    am = arch_model(y.values * scale, mean='Zero', vol=vol, p=p, q=q, dist='t')
    res = am.fit(disp='off')

    # 視窗內「過濾」波動：先除回 scale，再做下限保護
    sigma_series = res.conditional_volatility          # ndarray
    sigma_series = np.maximum(sigma_series / scale, 1e-12)
    log_sigma_series = np.log(sigma_series)
    log_sigma_series = pd.Series(log_sigma_series, index=y.index)\
                          .reindex(ret_window.index).ffill().bfill()

    # 一步前瞻：variance → sqrt → 除回 scale
    fvar = res.forecast(horizon=1, reindex=False).variance.values[-1, 0]
    sigma_next = float(np.sqrt(fvar) / scale)

    return log_sigma_series, sigma_next

################## 風險
def compute_var_es(mu, sigma, alpha,
                   dist=None,              # 'normal' 或 't'；若為 None 則依 nu 推斷
                   nu=None,                # dist='t' 時需要；dist='normal' 時忽略
                   price_base=None,        # 若給，會同時輸出價格層 VaR/ES
                   standardized_t=True     # t 分布是否以 Var=1 標準化
                   ):
    # """
    # 計算左尾 VaR/ES（報酬層 + 價格層）。
    # - mu, sigma: 可為 Series 或純量；若 Series，index 會自動對齊。
    # - alpha: 左尾機率（例如 0.05、0.01）。
    # - dist: 'normal' 或 't'。若 None：若 nu 是 None → 'normal'；否則 → 't'。
    # - nu:    t 分布自由度（>2）；dist='normal' 時忽略。
    # - standardized_t: True 時採用 Var=1 的 t-innovations（尺度 s = sqrt((nu-2)/nu)）。
    # - price_base: 若給（通常為 P_{t-1}），同時回傳 VaR_price / ES_price。
    # 回傳：DataFrame，含 VaR_ret、ES_ret（與可選 VaR_price、ES_price）。
    # """
    # ----- 推斷分布類型（維持舊版相容） -----
    if dist is None:
        dist = 't' if nu is not None else 'normal'
    dist = dist.lower()
    if dist not in ('normal', 'gaussian', 't', 'student', 'student-t'):
        raise ValueError("dist 必須為 'normal' 或 't'。")

    # ----- 對齊輸入 -----
    mu_s  = pd.Series(mu)
    sig_s = pd.Series(sigma)
    idx   = mu_s.index.union(sig_s.index)
    mu_s  = mu_s.reindex(idx)
    sig_s = sig_s.reindex(idx).clip(lower=0.0)  # 數值保險：σ >= 0
    price_s = pd.Series(price_base).reindex(idx) if price_base is not None else None

    if not (0.0 < alpha < 0.5):
        raise ValueError("alpha 應位於 (0, 0.5) 的左尾區間。")

    # ----- 常態分布 -----
    if dist in ('normal', 'gaussian'):
        z_alpha = norm.ppf(alpha)          # 負值
        phi     = norm.pdf(z_alpha)
        ES_std  = -phi / alpha             # 標準常態左尾 ES（均值0、σ=1）
        VaR_ret = mu_s + sig_s * z_alpha
        ES_ret  = mu_s + sig_s * ES_std

    # ----- t 分布 -----
    else:
        if nu is None:
            raise ValueError("dist='t' 需要提供 nu。")
        nu = float(nu)
        if nu <= 2:
            raise ValueError("t 分布計算 ES 需要 nu > 2（變異數有限）。")

        t_alpha = tdist.ppf(alpha, df=nu)     # 負值
        # 標準化尺度：Var=1 的 t-innovations 需乘 s = sqrt((nu-2)/nu)
        s = np.sqrt((nu - 2.0) / nu) if standardized_t else 1.0
        f_t = tdist.pdf(t_alpha, df=nu)

        # 標準型 t 的左尾 ES（均值0、尺度1）封閉解
        ES_std = - ((nu + t_alpha**2) / ((nu - 1.0) * alpha)) * f_t

        VaR_std = s * t_alpha
        ES_std  = s * ES_std

        VaR_ret = mu_s + sig_s * VaR_std
        ES_ret  = mu_s + sig_s * ES_std

    out = pd.DataFrame({'VaR_ret': VaR_ret, 'ES_ret': ES_ret}, index=idx)

    # ----- 價格層（若提供基準價） -----
    if price_s is not None:
        out['VaR_price'] = price_s * np.exp(out['VaR_ret'])
        out['ES_price']  = price_s * np.exp(out['ES_ret'])

    return out.reindex(mu_s.index)


# === 0) (可選) 判斷 t 殘差是否已標準化 Var≈1 ===
def check_t_standardization(eps, sigma) -> float:
    # """
    # 回傳標準化殘差 z = eps/sigma 的樣本變異。
    # ≈1 代表 standardized_t=True；若 ≈ ν/(ν-2) 代表未標準化。
    # """
    z = np.asarray(eps) / np.maximum(np.asarray(sigma), 1e-12)
    z = z[np.isfinite(z)]
    return float(np.var(z, ddof=1))


# ------------------ Config ------------------
ASSET_SYMBOL_ES1 = 'ES1'
ASSET_SYMBOL_VIX = 'VIX'
GROUP_DAY   = 'stock_day'
TARGET_START_STR = '2005-05-31'
TARGET_END_STR   = '2024-12-31'

WINDOW_DAY = 20  # 14 days
GARCH_WINDOW_DAY = 250
MODE = 'warm'           # 'warm' or 'refit'
EPOCHS_INIT = 8
EPOCHS_STEP = 1
BATCH_SIZE = 32
LR = 5e-4
LSTM_UNITS = 64
DROPOUT = 0.2
USE_HUBER = True
NORMALIZE_FEATURES = True  # z-score inputs per window

# ------------------ Features ------------------
feature_names = ['ES1_LN_RET','ES1_OPEN','ES1_HIGH','ES1_LOW','ES1_CLOSE','ES1_VOLUME','ES1_CLOSE_LN','VIX_CLOSE_LN']

feat_ES1 = build_feature_df(repo, GROUP_DAY, ASSET_SYMBOL_ES1)
feat_VIX = build_feature_df(repo, GROUP_DAY, ASSET_SYMBOL_VIX)

df_day = feat_ES1.join(feat_VIX, how='left')

# --- 讓索引成為 datetime（很重要） ---
df_day.index = pd.to_datetime(df_day.index, errors='coerce')
df_day = df_day.sort_index()
df_day = df_day.loc[df_day['ES1_IS_TRADING'] == 1]

# print(df_day.head())
# print(df_day.columns)

df_day.to_csv("./filtered_output/df_day.csv", index=True)


PRED_START = pd.to_datetime(TARGET_START_STR)
PRED_END   = pd.to_datetime(TARGET_END_STR)

# 把回測期限制在資料範圍內
data_start = df_day.index.min()
data_end   = df_day.index.max()
if PRED_START < data_start: PRED_START = data_start
if PRED_END   > data_end:   PRED_END   = data_end

# 若 PRED_START 不是可用交易日，推到 >= PRED_START 的第一個交易日
try:
    PRED_START = df_day.index[df_day.index.searchsorted(PRED_START)]
except Exception:
    # 若整段都沒資料，直接報錯
    raise RuntimeError("資料期間與回測期間沒有交集，請調整 TARGET_START/END。")


first_needed = PRED_START - pd.Timedelta(days=WINDOW_DAY)
es1_min = df_day.index.min()
ES1_close_series = df_day['ES1_CLOSE']
if es1_min > first_needed:
    raise RuntimeError(f"Insufficient history: need <= {first_needed}, have from {ES1_close_series.index.min()}")



# ------------------ Model ------------------

# 1) 準備母表（確保排序與期間）
# 全歷史的交易日索引（只留 ES1 開市）
dates_all = df_day.index
# print(dates_all)

# 回測區間（仍然要取，決定哪些 t 需要預測）
mask_period = (df_day.index >= PRED_START) & (df_day.index <= PRED_END)
dates_period = df_day.index[mask_period]


# 2) 方便取用的短名
feat_df = df_day  # 與舊程式一致
model: Optional[tf.keras.Model] = None
rows = []


# 3) 走索引的滑窗，不假設日期連續
#   第 i 筆要預測的是 dates[i]（以 t 表示），窗口是 dates[i-WINDOW_DAY : i)（只到 t-1）

for t in dates_period:
    # 1) 找出 t 在「全歷史」中的**位置**（不是在 dates_period 中的 i）
    pos = dates_all.searchsorted(t)   # 或：pos = df_day.index.get_loc(t)

    # 2) 用「位置」切出兩個視窗（用 take 或 iloc，千萬不要用 dates_all[a:b]）
    if pos < max(WINDOW_DAY, GARCH_WINDOW_DAY):
      continue  # 歷史不足就跳過

    win_idx   = np.arange(pos - WINDOW_DAY, pos)    # → [t-WINDOW, ..., t-1]
    garch_idx = np.arange(pos - GARCH_WINDOW_DAY, pos)  # → [t-GARCH,   ..., t-1]

    win_dates   = dates_all.take(win_idx)
    garch_dates = dates_all.take(garch_idx)

    # 目標日與 t-1
    t_minus_1 = dates_all[pos - 1]   # 直接往前一格 → t-1
    print(f"pos = {dates_all[pos]},  window = {win_dates[0]} -> {win_dates[-1]}")

#     print(f"start {t} ")
    # print(f"start {t} | pos {pos} |　t_minus_1 {t_minus_1}")

    # print(f"garch head={list(garch_dates[:5].date)}")
    # print(f"garch tail={list(garch_dates[-5:].date)}\n")
    # print(f"Xw_period head={list(win_dates[:5].date)}")
    # print(f"Xw_period tail={list(win_dates[-5:].date)}\n")

    # --- 視窗特徵 ---
    Xw = feat_df.loc[win_dates, feature_names].copy()
    if len(Xw) != WINDOW_DAY or Xw.isna().any().any():
        # 視窗內若有缺值，略過這天
        continue

    # --- GARCH（視窗內，只用到 t-1 的 return） ---
    # 你的 feature_names 中若名稱是 ES1_LN_RET 就用它；否則改用你實際欄名
    # garch_dates有300天做完garch後便做zcore存到log_sigma300，再將其取所需時間段出來
    ret_col = 'ES1_LN_RET' if 'ES1_LN_RET' in feat_df.columns else 'LN_RET'
    log_sigma_series, sigma_next = fit_vol_per_window(
        ret_window=feat_df.loc[garch_dates, ret_col], # 只含 ≤t-1
        mode='garch'  # 'harch' 也可
    )
    # print(len(log_sigma_series))

    mu300 = float(log_sigma_series.mean())
    sd300 = float(log_sigma_series.std(ddof=0))
    log_sigma300 = (log_sigma_series - mu300) / sd300
    Xw['LOG_SIGMA'] = log_sigma300.reindex(win_dates).ffill().fillna(0.0).values


    # --- 每窗 z-score（避免洩漏）---
    if NORMALIZE_FEATURES:
        # 排除已在 z 範圍內的報酬欄（可保留或一起 z-score，影響不大）
        cols_to_norm = ['ES1_OPEN','ES1_HIGH','ES1_LOW','ES1_CLOSE','ES1_VOLUME','ES1_CLOSE_LN','VIX_CLOSE_LN']
        mu = Xw[cols_to_norm].mean(axis=0)
        sd = Xw[cols_to_norm].std(axis=0).replace(0.0, np.nan)
        Xw[cols_to_norm] = (Xw[cols_to_norm] - mu) / sd

        Xw = Xw.fillna(0.0)

    names_this = list(Xw.columns)
    N_FEATURES_THIS = len(names_this)
    X_t = Xw[names_this].values.reshape(1, WINDOW_DAY, N_FEATURES_THIS).astype(np.float32) # ≤t-1

    # --- 目標：下一日「標準化 ln 價」---
    ln_tm1 = float(feat_df.loc[t_minus_1, 'ES1_CLOSE_LN'])
    ln_t   = float(feat_df.loc[t,'ES1_CLOSE_LN'])

    # 目標標準化所需的均值/標準差（用視窗內 ln 價）
    mu_ln  = float(feat_df.loc[win_dates, 'ES1_CLOSE_LN'].mean())
    sig_ln = float(feat_df.loc[win_dates, 'ES1_CLOSE_LN'].std(ddof=0) + 1e-12)
    y_z_t  = np.array([(ln_t - mu_ln)/sig_ln], dtype=np.float32)

    # --- 先 predict 再 fit（walk-forward）---
    if model is not None:
        z_hat      = float(model.predict(X_t, verbose=0).ravel()[0])   # 預測的 z(lnP)  # 只吃 ≤t-1 特徵
        yhat_ln_t  = z_hat * sig_ln + mu_ln                # 反標準化成 lnP ⇒ E[ln P_t | 𝔽_{t-1}]
        price_pred = float(np.exp(yhat_ln_t))                  #⇒ P̂_t | 𝔽_{t-1}
        r_hat      = yhat_ln_t - ln_tm1                 # 預測對數報酬⇒ E[r_t | 𝔽_{t-1}]
    else:
        yhat_ln_t = np.nan; price_pred = np.nan; r_hat = np.nan

    # --- 模型建/續訓 ---
    if (model is None) or (MODE == 'refit'):
        model = build_small_lstm(WINDOW_DAY, N_FEATURES_THIS)  # 注意是 DAY 的窗口長度
        epochs_here = EPOCHS_INIT
    else:
        epochs_here = EPOCHS_STEP

    hist = model.fit(X_t, y_z_t, epochs=epochs_here, batch_size=1, shuffle=False, verbose=0)
    last_loss = float(hist.history['loss'][-1])

    # --- 真實值 ---
    p_tm1 = float(feat_df.loc[t_minus_1, 'ES1_CLOSE'])
    p_t   = float(feat_df.loc[t, 'ES1_CLOSE'])
    r_t   = float(feat_df.loc[t, 'ES1_LN_RET'])


    rows.append({
        'DATE'      : t,
        'ln_true'   : ln_t,
        'ln_pred'   : yhat_ln_t,
        'ret_true'  : r_t,
        'ret_pred'  : r_hat,
        'price_true': p_t,
        'price_pred': price_pred,
        'sigma_next': sigma_next,
        'train_loss': last_loss,
    })


# # ------------------ Evaluation ------------------
######################################################################################################
# ##########  風險
# 先建 df_out
df_out = pd.DataFrame(rows).set_index("DATE").sort_index()

# 價格 VaR 的基準：P_{t-1} 對齊到 t
df_out['price_base_aligned'] = df_out['price_true'].shift(1)
df_out = df_out.dropna(subset=['price_base_aligned'])

# 參數
ALPHAS = [0.05, 0.01]
NU     = 6.0
STD_T  = True

mu_series    = df_out['ret_pred']      # μ̂_t（as-of t-1 對 t）
sigma_series = df_out['sigma_next']    # σ̂_t（as-of t-1 對 t）
price_base   = df_out['price_base_aligned']  # P_{t-1}（對齊到 t）

for a in ALPHAS:
    res = compute_var_es(mu_series, sigma_series, alpha=a,
                     dist='t', nu=6, price_base=price_base, standardized_t=True)
    tag = f'{int((1 - a)*100):02d}'
    df_out[f'VaR_ret_{tag}'] = res['VaR_ret']
    df_out[f'ES_ret_{tag}']  = res['ES_ret']
    if 'VaR_price' in res.columns:
        df_out[f'VaR_price_{tag}'] = res['VaR_price']
    if 'ES_price' in res.columns:
        df_out[f'ES_price_{tag}']  = res['ES_price']

# 報酬層違規
df_out['viol_95'] = (df_out['ret_true'] < df_out['VaR_ret_95']).astype(int)
df_out['viol_99'] = (df_out['ret_true'] < df_out['VaR_ret_99']).astype(int)

# 快速違規率
viol_rate_95 = float(df_out['viol_95'].mean())
viol_rate_99 = float(df_out['viol_99'].mean())

# 逐日輸出（修正漏引號）
cols_daily = [
    'ln_true','ln_pred',
    'ret_true','price_true',
    'sigma_next',   # ← 修正這裡
    'VaR_ret_95','ES_ret_95','viol_95',
    'VaR_ret_99','ES_ret_99','viol_99',
    'VaR_price_95','ES_price_95','VaR_price_99','ES_price_99'
]
df_export = df_out[[c for c in cols_daily if c in df_out.columns]].copy()
# 若你有價格層級欄位，也一起輸出
for extra in ['VaR_price_95','ES_price_95','VaR_price_99','ES_price_99']:
    if extra in df_out.columns:
        cols_daily.append(extra)

df_export = df_out[[c for c in cols_daily if c in df_out.columns]].copy()
# daily_path = os.path.join(output_dir, "var_es_daily.csv")
# df_export.to_csv(daily_path, index=True)  # index=DATE

# 6b) 摘要矩陣（風控報告用）
var_es_matrics = pd.DataFrame({
    'Metric': [
        'Mean_ret_true', 'Std_ret_true',
        'Mean_VaR_ret_95', 'Mean_ES_ret_95',
        'Mean_VaR_ret_99', 'Mean_ES_ret_99',
        'Viol_rate_95', 'Viol_count_95',
        'Viol_rate_99', 'Viol_count_99',
    ],
    'Value': [
        float(df_out['ret_true'].mean()),
        float(df_out['ret_true'].std(ddof=1)),
        float(df_out['VaR_ret_95'].mean()),
        float(df_out['ES_ret_95'].mean()),
        float(df_out['VaR_ret_99'].mean()),
        float(df_out['ES_ret_99'].mean()),
        viol_rate_95, int(df_out['viol_95'].sum()),
        viol_rate_99, int(df_out['viol_99'].sum()),
    ]
})

###############################################################################################################
################################################################################################################
def compute_direction_metrics(df, col_true='ret_true', col_pred='ret_pred', eps=0.0):
    # """
    # 方向性評估：分別計算實際上漲/下跌時的命中率與混淆矩陣。
    # - eps: 將 |ret| <= eps 視為「小波動」，可選擇剔除（默認 0 不剔除）。
    # 回傳：metrics(dict), confusion(dict), mask_info(dict)
    # """
    # 1) 安全遮罩（有限實數）
    m = np.isfinite(df[col_true]) & np.isfinite(df[col_pred])
    d = df.loc[m, [col_true, col_pred]].copy()
    if d.empty:
        return {}, {}, {'n_used': 0, 'n_total': len(df)}

    # 2) 可選：忽略極小真實變動（避免雜訊），僅在 eps > 0 時啟用
    if eps > 0:
        keep = d[col_true].abs().values > eps
        d = d.loc[keep]
    if d.empty:
        return {}, {}, {'n_used': 0, 'n_total': len(df)}

    # 3) 方向標籤：上漲=1，下跌=0
    y_true = (d[col_true].values > 0).astype(int)
    y_pred = (d[col_pred].values > 0).astype(int)

    # 4) 混淆矩陣計數
    TP = int(((y_pred == 1) & (y_true == 1)).sum())  # 預測漲且實際漲
    TN = int(((y_pred == 0) & (y_true == 0)).sum())  # 預測跌且實際跌
    FP = int(((y_pred == 1) & (y_true == 0)).sum())
    FN = int(((y_pred == 0) & (y_true == 1)).sum())

    # 基本計數
    actual_up   = int((y_true == 1).sum())
    actual_down = int((y_true == 0).sum())
    pred_up     = int((y_pred == 1).sum())
    pred_down   = int((y_pred == 0).sum())
    n = len(d)

    # 5) 指標（加上小心分母為 0 的情形）
    def safe_div(a, b): return float(a) / float(b) if b else np.nan

    acc_overall   = safe_div(TP + TN, n)
    acc_up        = safe_div(TP, actual_up)     # 「實際漲」時的命中率
    acc_down      = safe_div(TN, actual_down)   # 「實際跌」時的命中率
    precision_up  = safe_div(TP, TP + FP)       # 預測漲的精確率
    recall_up     = acc_up                      # = TP/(TP+FN)
    specificity   = acc_down                    # = TN/(TN+FP)
    bal_acc       = np.nanmean([recall_up, specificity])  # (敏感度+特異度)/2

    # Matthews Correlation Coefficient（方向相關的穩健指標）
    denom = np.sqrt((TP+FP)*(TP+FN)*(TN+FP)*(TN+FN))
    mcc = ((TP*TN - FP*FN)/denom) if denom else np.nan

    metrics = {
        'n_used': n,
        'TP': TP,
        'FP': FP,
        'TN': TN,
        'FN': FN,
        'n_total_in_df': len(df),
        'actual_up': actual_up,
        'actual_down': actual_down,
        'pred_up': pred_up,
        'pred_down': pred_down,
        'acc_overall': acc_overall,
        'acc_up': acc_up,                 # 教授關注：實際上漲時的命中率
        'acc_down': acc_down,             # 教授關注：實際下跌時的命中率
        'precision_up': precision_up,
        'recall_up': recall_up,
        'specificity_down': specificity,
        'balanced_accuracy': bal_acc,
        'mcc': mcc,
        'eps': eps
    }

    confusion = {'TP': TP, 'FP': FP, 'TN': TN, 'FN': FN}
    mask_info = {'n_used': n, 'n_total': len(df), 'eps': eps}
    return metrics, confusion, mask_info

############   回歸預測
df_pred_sw = df_out.copy()

# 1) 安全遮罩（同列對齊；你已移除 *_aligned）
is_finite_ret   = np.isfinite(df_pred_sw['ret_true'])   & np.isfinite(df_pred_sw['ret_pred'])
is_finite_price = np.isfinite(df_pred_sw['price_true']) & np.isfinite(df_pred_sw['price_pred'])

n_ret   = int(is_finite_ret.sum())
n_price = int(is_finite_price.sum())

# 2) Returns 指標
if n_ret > 0:
    y_true_ret = df_pred_sw.loc[is_finite_ret, 'ret_true'].values
    y_pred_ret = df_pred_sw.loc[is_finite_ret, 'ret_pred'].values

    mae_val  = float(mean_absolute_error(y_true_ret, y_pred_ret))
    rmse_val = float(np.sqrt(mean_squared_error(y_true_ret, y_pred_ret)))
    r2_ret   = float(r2_score(y_true_ret, y_pred_ret)) if n_ret > 1 else np.nan

    # 方向性
    y_true_dir = (y_true_ret > 0).astype(int)
    y_pred_dir = (y_pred_ret > 0).astype(int)
    precision  = float(precision_score(y_true_dir, y_pred_dir, zero_division=0))
    recall     = float(recall_score(y_true_dir, y_pred_dir, zero_division=0))
    f1         = float(f1_score(y_true_dir, y_pred_dir, zero_division=0))
    acc_dir    = float((y_true_dir == y_pred_dir).mean())

    # 簽策略：ret_true 為 ln-return → 用「對數世界」累積
    strat = np.sign(y_pred_ret) * y_true_ret
    ann_factor  = 252  # 日資料；若改非日頻，記得相應調整
    total_return = float(np.exp(strat.sum()) - 1.0)
    ann_return   = float(np.exp(strat.mean() * ann_factor) - 1.0)
    equity       = np.exp(strat.cumsum())           # 權益曲線（log→level）
    roll_max     = np.maximum.accumulate(equity)
    max_dd       = float((equity / roll_max - 1.0).min())
else:
    mae_val = rmse_val = r2_ret = precision = recall = f1 = acc_dir = total_return = ann_return = max_dd = np.nan

# 3) Price 指標（與 returns 分開判斷）
if n_price > 0:
    y_true_px = df_pred_sw.loc[is_finite_price, 'price_true'].values
    y_pred_px = df_pred_sw.loc[is_finite_price, 'price_pred'].values

    mae_price  = float(mean_absolute_error(y_true_px, y_pred_px))
    rmse_price = float(np.sqrt(mean_squared_error(y_true_px, y_pred_px)))
    r2_price   = float(r2_score(y_true_px, y_pred_px)) if n_price > 1 else np.nan
else:
    mae_price = rmse_price = r2_price = np.nan

# 4) 平均訓練損失
avg_loss = float(np.nanmean(df_pred_sw['train_loss'])) if len(df_pred_sw) else np.nan

# 5) 匯總表（去除重複鍵，補充樣本數與 Directional Accuracy）
metrics_df = pd.DataFrame({
    'asset':         [ASSET_SYMBOL_ES1],
    'group':         [GROUP_DAY],
    'start':         [TARGET_START_STR],
    'end':           [TARGET_END_STR],
    'n_ret':         [n_ret],
    'n_price':       [n_price],
    'mae':           [mae_val],
    'rmse':          [rmse_val],
    'R2_ret':        [r2_ret],
    'mae_price':     [mae_price],
    'rmse_price':    [rmse_price],
    'R2_price':      [r2_price],
    # 'DA':            [acc_dir],          # Directional Accuracy
    'F1_Score':      [f1],
    'Precision':     [precision],
    'Recall':        [recall],
    'avg_loss':      [avg_loss],
    'total_return':  [total_return],
    'ann_return':    [ann_return],
    'max_drawdown':  [max_dd],
})
##混淆矩正
# 假設 df_out 內 ret_true / ret_pred 為同列對齊的對數報酬
metrics_strict, conf_strict, info_strict = compute_direction_metrics(df_out, 'ret_true', 'ret_pred', eps=0.0)
metrics_eps,    conf_eps,    info_eps    = compute_direction_metrics(df_out, 'ret_true', 'ret_pred', eps=0.0005)
metrics_strict_df = pd.DataFrame([metrics_strict])   # 嚴格版（不忽略微小波動）



# ------------------ Save CSVs ------------------
# === 匯出檔案 ===
output_dir = "./LSTM_diagnostics"
os.makedirs(output_dir, exist_ok=True)

stem = f"{ASSET_SYMBOL_ES1.lower()}_lstm_day_sliding_multifeat_nogarch"
os.makedirs("./LSTM_diagnostics", exist_ok=True)

out_csv = f"./LSTM_diagnostics/{stem}_{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.csv"
df_pred_sw.to_csv(out_csv, float_format='%.10f')

metrics_vertical =  metrics_df.T.reset_index()
metrics_vertical.columns = ["metric", "value"]
metrics_csv = f"./LSTM_diagnostics/{stem}_metrics_{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.csv"
metrics_vertical.to_csv(metrics_csv, index=False)

metrics_strict_vertical = metrics_strict_df.T.reset_index()
metrics_strict_vertical.columns = ["metric", "value"]
metrics_strict_csv = f"./LSTM_diagnostics/{stem}_confusion_metrics_{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.csv"
metrics_strict_vertical.to_csv(metrics_strict_csv, index=False)

###############  風險
daily_path = os.path.join(output_dir, "var_es_daily.csv")
df_export.to_csv(daily_path, index=True)  # index=DATE

summary_path = os.path.join(output_dir, "var_es_matrics.csv")
var_es_matrics.to_csv(summary_path, index=False)

print("Saved:", out_csv)
print("Metrics:", metrics_csv)

# ------------------ Figures (3+1) ------------------
fig1 = f"./LSTM_diagnostics/{stem}_price_{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.png"
fig2 = f"./LSTM_diagnostics/{stem}_returns_{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.png"
fig3 = f"./LSTM_diagnostics/{stem}_scatter_{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.png"
fig4 = f"./LSTM_diagnostics/{stem}_trainloss_price_{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.png"
fig5 = f"./LSTM_diagnostics/{stem}_VaR_ES_price_{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.png"
fig6 = f"./LSTM_diagnostics/{stem}_VaR_ES_returns_{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.png"
fig7 = f"./LSTM_diagnostics/{stem}_VaR_ES_scatter_{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.png"

plt.figure(figsize=(12,6))
plt.plot(df_pred_sw.index, df_pred_sw['price_true'], label='True Price',linewidth=0.8, alpha=0.7)
plt.plot(df_pred_sw.index, df_pred_sw['price_pred'], label='Predicted Price', linestyle='--',linewidth=0.8, alpha=0.7)
plt.xlabel('Time'); plt.ylabel('Price'); plt.title(f"{ASSET_SYMBOL_ES1} Price"); plt.legend(); plt.grid(True)
plt.savefig(fig1, dpi=300); plt.close()

plt.figure(figsize=(12,6))
plt.plot(df_pred_sw.index, df_pred_sw['ret_true'], label='True Return', linewidth=0.8,alpha=0.7)
plt.plot(df_pred_sw.index, df_pred_sw['ret_pred'], label='Predicted Return', linewidth=0.8,alpha=0.7)
plt.axhline(0, color='black', linewidth=1)
plt.xlabel('Time'); plt.ylabel('Log Return'); plt.title(f"{ASSET_SYMBOL_ES1} Returns"); plt.legend(); plt.grid(True)
plt.savefig(fig2, dpi=300); plt.close()

plt.figure(figsize=(6,6))
plt.scatter(df_pred_sw['ret_true'], df_pred_sw['ret_pred'], alpha=0.5)
plt.axhline(0, color='black', linewidth=1); plt.axvline(0, color='black', linewidth=1)
plt.xlabel('True Return'); plt.ylabel('Predicted Return'); plt.title(f"{ASSET_SYMBOL_ES1} True vs Predicted Returns"); plt.grid(True)
plt.savefig(fig3, dpi=300); plt.close()

# Price + Train Loss
fig, ax1 = plt.subplots(figsize=(12,6))
ax1.plot(df_pred_sw.index, df_pred_sw['price_true'], label='True Price',linewidth=0.8, alpha=0.7)
ax1.plot(df_pred_sw.index, df_pred_sw['price_pred'], label='Predicted Price', linestyle='--', linewidth=0.8,alpha=0.9)
ax1.set_xlabel("Time"); ax1.set_ylabel("Price")
ax2 = ax1.twinx()
ax2.plot(df_pred_sw.index, df_pred_sw['train_loss'], label='Train Loss', alpha=0.6, color='green')
ax2.set_ylabel("Training Loss")
ax1.grid(True, which='both', axis='both', alpha=0.2)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout(); plt.savefig(fig4, dpi=300); plt.close()


############## 風險
# === 圖 1: 價格 vs VaR/ES (95%) ===
plt.figure(figsize=(12,6))
plt.plot(df_out.index, df_out['price_true'],
         label='True Price', color='black', linewidth=0.8)
plt.plot(df_out.index, df_out['VaR_price_95'],
         label='VaR Price 95%', color='red', linewidth=0.5, alpha=0.7)
plt.plot(df_out.index, df_out['ES_price_95'],
         label='ES Price 95%', color='darkred', linestyle='--', linewidth=0.8, alpha=0.7)
plt.fill_between(df_out.index, 0, df_out['VaR_price_95'],
                 color='red', alpha=0.05)

plt.xlabel("Time"); plt.ylabel("Price")
plt.title(f"{ASSET_SYMBOL_ES1} Price vs VaR/ES (95%)")
plt.legend(); plt.grid(True)
plt.savefig(fig5, dpi=300); plt.close()

# === 圖 2: 報酬 vs VaR/ES (95%) ===
plt.figure(figsize=(12,6))
plt.plot(df_out.index, df_out['ret_true'], label='True Return', color='blue', alpha=0.6, linewidth=0.6)
plt.plot(df_out.index, df_out['VaR_ret_95'], label='VaR 95%', color='red', linewidth=0.6, alpha=0.7)
plt.plot(df_out.index, df_out['ES_ret_95'], label='ES 95%', color='darkred', linestyle='--', linewidth=0.8, alpha=0.7)
viol = df_out['ret_true'] < df_out['VaR_ret_95']
plt.scatter(df_out.index[viol], df_out['ret_true'][viol],
            color='red', marker='x', s=20, linewidths=0.8, label='Violation')
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.xlabel("Time"); plt.ylabel("Log Return")
plt.title(f"{ASSET_SYMBOL_ES1} Return vs VaR/ES (95%)")
plt.legend(); plt.grid(True)
plt.savefig(fig6, dpi=300); plt.close()

# === 圖 3: 散點回測圖 (VaR vs True Return, 95%) ===
plt.figure(figsize=(6,6))
plt.scatter(df_out['VaR_ret_95'], df_out['ret_true'],
            alpha=0.4, color='blue', s=10)  # 點數縮小、透明度提高
plt.plot([df_out['VaR_ret_95'].min(), df_out['VaR_ret_95'].max()],
         [df_out['VaR_ret_95'].min(), df_out['VaR_ret_95'].max()],
         color='red', linestyle='--', linewidth=1.0)
plt.xlabel("VaR_ret_95"); plt.ylabel("True Return")
plt.title(f"{ASSET_SYMBOL_ES1} VaR Backtesting (95%)")
plt.grid(True)
plt.savefig(fig7, dpi=300); plt.close()

print('Figures:', fig1, fig2, fig3, fig4)


Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.


Num GPUs: 1 [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✓ memory growth set
Num GPUs: 1 [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✓ memory growth set
是否可用GPU: True
使用中的裝置: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✓ GPU 初始化流程完成
pos = 2005-05-31 00:00:00,  window = 2005-05-02 00:00:00 -> 2005-05-27 00:00:00
pos = 2005-06-01 00:00:00,  window = 2005-05-03 00:00:00 -> 2005-05-31 00:00:00
pos = 2005-06-02 00:00:00,  window = 2005-05-04 00:00:00 -> 2005-06-01 00:00:00
pos = 2005-06-03 00:00:00,  window = 2005-05-05 00:00:00 -> 2005-06-02 00:00:00
pos = 2005-06-06 00:00:00,  window = 2005-05-06 00:00:00 -> 2005-06-03 00:00:00
pos = 2005-06-07 00:00:00,  window = 2005-05-09 00:00:00 -> 2005-06-06 00:00:00
pos = 2005-06-08 00:00:00,  window = 2005-05-10 00:00:00 -> 2005-06-07 00:00:00
pos = 2005-06-09 00:00:00,  window = 2005-05-11 00:00:00 -> 2005-06-08 00:00:00
pos = 2005-06-10 00:00:00,  window = 2005-05-12 00:00:00 ->